In [ ]:
# SETUP
# DICOM/RTSTRUCT and 3D Slicer MRB to paired NRRD converter for Google Colab
#
# WHAT GOES IN
# The notebook reads two input formats and tells them apart by itself, so
# there is no format switch to set. Both formats can be mixed in one root.
# - DICOM: a DICOM scan is not one file but a folder of many files, usually
#   one file per image slice. The tumour outline (RTSTRUCT) is another file in
#   the same patient folder.
# - MRB: a .mrb file ("Medical Reality Bundle") is a whole 3D Slicer scene
#   saved as a single file. It holds the scan and the segmentation drawn on it.
# A "root" folder is the top-level folder you point the notebook to. Each item
# directly inside a root counts as one patient:
# - a folder = one patient folder. The notebook searches it and every folder
#   inside it for DICOM files and .mrb files;
# - a .mrb file = one patient, named after the file (san.mrb -> patient "san").
#
# SETTINGS TO EDIT
# - Runtime: choose a high-RAM runtime (>50 GB working memory recommended).
# - dicom_root, dicom_root2, dicom_root3: Google Drive paths of up to three root
#   folders. Despite the names, each root may hold DICOM patient folders, .mrb
#   files, or both. All roots found are processed together. Unused or missing
#   roots are ignored. Patient names (folder names, and .mrb file names without
#   ".mrb") must be unique across all roots.
#   Google Drive folders with many files can make RAM usage spike to 30+ GB
#   for several minutes while the files are first read. To reduce this, spread
#   the patient folders over the three roots, always moving whole patient
#   folders.
# - output_root: folder for the converted paired NRRD image and mask files.
#   Notebook 2 INPUT_ROOT must use the
#   same path. Each patient produces two files named only by the patient ID:
#   the scan (<patient>.nrrd, e.g. 155.nrrd) and its tumour mask, an image
#   marking the tumour (<patient>_mask.nrrd, e.g. 155_mask.nrrd). Scans
#   without a usable outline are saved as <patient>.nrrd in a separate
#   folder, created automatically as <output_root>_unpaired. Notebook 2 needs
#   one scan-mask pair per patient, so if a patient has more than one usable
#   outline or segmentation, only the first is saved and the rest are listed
#   at the end of the run. Empty output_root of files from earlier runs
#   before running, so that Notebook 2 sees only the new files.
# - MENINGIOMA_KEYWORDS and SHORT_EXACT_CODES: an RTSTRUCT holds several named
#   outlines (ROIs, "regions of interest"); an MRB segmentation holds several
#   named segments. The same rules pick the tumour in both. Check the names in
#   3D Slicer and list the tumour names here, in lowercase:
#     KEYWORDS match anywhere in the name ("menin" matches "Meningioma_L").
#     SHORT_EXACT_CODES must equal the whole name ("tu" matches "TU" only).
#   Names with "skull" in them are always excluded. All matching outlines or
#   segments are merged into one mask. Check every generated mask visually.
# - MRB_PICK_UNNAMED_SEGMENT: segments never renamed in Slicer keep default
#   names (Segment_1, Segment_2, ...) that no name rule can match. When True
#   and no segment name matches the rules, the notebook skips box-shaped
#   background segments (the "other" seed left by Slicer's "Grow from seeds"
#   tool; it fills at least MRB_BACKGROUND_FILL of the box around it) and uses
#   the remaining default-named segment, but only if exactly one remains.
#   Otherwise it writes no mask. Set to False to use names only.



# Install dependencies
!pip -q install pynrrd pydicom SimpleITK rt-utils numpy

import os, re, warnings, shutil, tempfile, zipfile
import xml.etree.ElementTree as ET
from urllib.parse import unquote
import numpy as np
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder

# Reduce SimpleITK log noise
sitk.ProcessObject_SetGlobalWarningDisplay(False)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Paths: edit before running
dicom_root  = '/content/drive/MyDrive/DICOM_files_1'
dicom_root2  = '/content/drive/MyDrive/DICOM_files_2'
dicom_root3  = '/content/drive/MyDrive/DICOM_files_3'
output_root = '/content/drive/MyDrive/DATA/images_and_masks'
output_root_unpaired = output_root.rstrip('/\\') + '_unpaired'
os.makedirs(output_root, exist_ok=True)
os.makedirs(output_root_unpaired, exist_ok=True)
print(f"Paired NRRDs: {output_root}")
print(f"Image-only NRRDs: {output_root_unpaired}")
legacy_masks = [f for f in os.listdir(output_root) if f.lower().endswith('mask.seg.nrrd')]
if legacy_masks:
    warnings.warn(
        f'{len(legacy_masks)} legacy MASK.seg.nrrd file(s) already exist in output_root. '
        'Move or remove them before Notebook 2 to avoid duplicate masks.'
    )
earlier_nrrds = [f for f in os.listdir(output_root) if f.lower().endswith('.nrrd')]
if earlier_nrrds:
    warnings.warn(
        f'{len(earlier_nrrds)} .nrrd file(s) from an earlier run are already in output_root. '
        'Files with the same names will be overwritten; the others stay and Notebook 2 '
        'will read them too. Empty the folder first for a clean set.'
    )

# Edit these ROI/segment names after inspecting representative studies in 3D Slicer.
MENINGIOMA_KEYWORDS = [
    "meningioma", "meningima", "mening2", "mening", "menin",
    "menmri", "menig", "tumor", "tum ", "adenoma", "herwin",
    "new volume menin", "postop", "men wrh", "men t1cc",
    "men po", "target",
]
SHORT_EXACT_CODES = {
    "m", "me", "men", "mg", "men2", "tu", "tum",
    "men po", "men wrh", "men t1cc", "tu pp", "tu2",
}

# MRB files: what to do when segments still have Slicer's default names.
MRB_PICK_UNNAMED_SEGMENT = True
MRB_BACKGROUND_FILL = 0.80   # 0.80 = fills 80% or more of its box -> background

def is_target_roi(name: str) -> bool:
    n = str(name or '').strip().lower()
    return bool(n) and 'skull' not in n and (
        n in SHORT_EXACT_CODES or any(term in n for term in MENINGIOMA_KEYWORDS)
    )

# Helpers

# Use all available CPU threads inside ITK/SimpleITK ops
sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(max(1, os.cpu_count() or 1))

# Files with these endings are never DICOM; skipping them saves reading time.
NON_DICOM_ENDINGS = (
    ".mrb", ".mrml", ".nrrd", ".nhdr", ".nii", ".gz", ".zip", ".mha", ".mhd",
    ".vtk", ".vtp", ".stl", ".png", ".jpg", ".jpeg", ".txt", ".csv", ".json",
    ".xml", ".pdf", ".ipynb", ".py",
)

def might_be_dicom(filename):
    return not filename.lower().endswith(NON_DICOM_ENDINGS)

def list_all_dicoms(root_dir):
    """Find DICOM files, including files without a .dcm extension."""
    out = []
    for dp, _, files in os.walk(root_dir):
        for f in files:
            if not might_be_dicom(f):
                continue
            fp = os.path.join(dp, f)
            if f.lower().endswith(".dcm"):
                out.append(fp)
                continue
            try:
                ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
                if getattr(ds, "SOPClassUID", None):
                    out.append(fp)
            except Exception:
                pass
    return out

def get_tags_quick(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    return {
        "SOPInstanceUID": getattr(ds, "SOPInstanceUID", None),
        "SeriesInstanceUID": getattr(ds, "SeriesInstanceUID", None),
        "InstanceNumber": getattr(ds, "InstanceNumber", None),
        "ImagePositionPatient": getattr(ds, "ImagePositionPatient", None),
        "ImageOrientationPatient": getattr(ds, "ImageOrientationPatient", None),
        "Modality": getattr(ds, "Modality", None),
        "SeriesDescription": getattr(ds, "SeriesDescription", None),
    }

def index_series_files_by_uid(all_dcms):
    series_map = {}
    for fp in all_dcms:
        try:
            uid = get_tags_quick(fp)["SeriesInstanceUID"]
            if uid:
                series_map.setdefault(uid, []).append(fp)
        except Exception:
            continue
    for uid in list(series_map.keys()):
        series_map[uid] = sorted(series_map[uid])
    return series_map

def sort_slice_files(files):
    """Sort slices by position, then instance number and filename."""
    records = []
    for fp in files:
        try:
            ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)
            inst = getattr(ds, "InstanceNumber", None)

            if iop and len(iop) == 6 and ipp is not None:
                row = np.array(iop[0:3], dtype=float)
                col = np.array(iop[3:6], dtype=float)
                normal = np.cross(row, col)
                t = float(np.dot(np.array(ipp, dtype=float), normal))
            else:
                t = None

            records.append((fp, t, inst))
        except Exception:
            records.append((fp, None, None))

    if any(r[1] is not None for r in records):
        records.sort(key=lambda r: (float('inf') if r[1] is None else r[1]))
    elif any(r[2] is not None for r in records):
        records.sort(key=lambda r: (float('inf') if r[2] is None else r[2]))
    else:
        records.sort(key=lambda r: r[0])

    return [r[0] for r in records]

def load_sitk_from_files(files_sorted):
    if not files_sorted:
        raise RuntimeError("Empty file list for volume load.")
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(files_sorted)
    return reader.Execute()

def _save_nrrd(image_sitk, path, compress=True):
    writer = sitk.ImageFileWriter()
    writer.SetFileName(path)
    writer.SetImageIO("NrrdImageIO")
    writer.UseCompressionOn() if compress else writer.UseCompressionOff()
    writer.Execute(image_sitk)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"File not written or empty: {path}")

def is_rtstruct(path):
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        return (getattr(ds, "SOPClassUID", "") == "1.2.840.10008.5.1.4.1.1.481.3") or \
               (getattr(ds, "Modality", "") == "RTSTRUCT")
    except Exception:
        return False

def list_rtstruct_files(root):
    """Find RTSTRUCT files, regardless of extension."""
    out = []
    for dp, _, files in os.walk(root):
        for f in files:
            if not might_be_dicom(f):
                continue
            fp = os.path.join(dp, f)
            if is_rtstruct(fp):
                out.append(fp)
    return sorted(out)

def get_referenced_series_uids_from_rtstruct(rtstruct_path):
    try:
        ds = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
    except Exception as e:
        warnings.warn(f"Failed to read RTSTRUCT {rtstruct_path}: {e}")
        return set()
    out = set()
    try:
        for rfr in getattr(ds, "ReferencedFrameOfReferenceSequence", []):
            for rstud in getattr(rfr, "RTReferencedStudySequence", []):
                for rser in getattr(rstud, "RTReferencedSeriesSequence", []):
                    uid = getattr(rser, "SeriesInstanceUID", None)
                    if uid:
                        out.add(uid)
    except Exception:
        pass
    return out

def choose_uid_with_most_files(candidates, series_map):
    best_uid, best_n = None, -1
    for uid in candidates:
        n = len(series_map.get(uid, []))
        if n > best_n:
            best_uid, best_n = uid, n
    return best_uid

def choose_fallback_uid(series_map):
    best_uid, best_n = None, -1
    for uid, files in series_map.items():
        n = len(files)
        if n > best_n:
            best_uid, best_n = uid, n
    return best_uid

def numpy_to_sitk_like(ref_img: sitk.Image, arr: np.ndarray) -> sitk.Image:
    img = sitk.GetImageFromArray(arr)  # (z,y,x)
    img.SetSpacing(ref_img.GetSpacing())
    img.SetOrigin(ref_img.GetOrigin())
    img.SetDirection(ref_img.GetDirection())
    return img

def make_temp_series_dir(files_sorted):
    """Copy one DICOM series to a temporary directory."""
    tmpdir = tempfile.mkdtemp(prefix="rtseries_")
    for i, src in enumerate(files_sorted, 1):
        dst = os.path.join(tmpdir, f"{i:06d}.dcm")
        shutil.copy2(src, dst)
    return tmpdir

def align_mask_axes(mask_np, target_shape):
    """Match common mask axis orders to target (z, y, x)."""
    if mask_np.shape == target_shape:
        return mask_np, "identity"
    for perm in [(2,0,1), (2,1,0), (1,2,0), (0,2,1), (1,0,2)]:
        try:
            cand = np.transpose(mask_np, perm)
            if cand.shape == target_shape:
                return cand, f"transpose{perm}"
        except Exception:
            pass
    return None, None

# MRB helpers
# An MRB is a zip archive holding a scene file (.mrml) that lists every item
# in the Slicer scene, plus a Data folder with the files: the scan (usually
# .nrrd) and the segmentation (.seg.nrrd).

IMAGE_FILE_ENDINGS = (".nrrd", ".nhdr", ".nii", ".nii.gz", ".mha", ".mhd")
DEFAULT_SEGMENT_NAME = re.compile(r"segment[_ ]?\d+", re.IGNORECASE)

def list_mrb_files(folder):
    """Find .mrb files in a folder and every folder inside it."""
    out = []
    for dp, _, files in os.walk(folder):
        out.extend(os.path.join(dp, f) for f in files if f.lower().endswith(".mrb"))
    return sorted(out)

def image_grid(path):
    """Read only the header of an image file: (size, spacing, direction)."""
    reader = sitk.ImageFileReader()
    reader.SetFileName(path)
    reader.ReadImageInformation()
    return (tuple(reader.GetSize()),
            tuple(round(s, 3) for s in reader.GetSpacing()),
            tuple(round(d, 3) for d in reader.GetDirection()))

def _find_file(folder, filename):
    for dp, _, files in os.walk(folder):
        if filename in files:
            return os.path.join(dp, filename)
    return None

def _mrml_refs(node):
    """Node references as {role: [node ids]}, e.g. {'storage': ['vtkMRML...1']}."""
    refs = {}
    for part in (node.get("references") or "").split(";"):
        role, _, ids = part.partition(":")
        if role.strip() and ids.strip():
            refs[role.strip()] = ids.split()
    if "storage" not in refs and node.get("storageNodeRef"):  # older Slicer scenes
        refs["storage"] = node.get("storageNodeRef").split()
    return refs

def read_mrb_scene(scene_dir):
    """List the scans and segmentations in an unpacked MRB.

    Each item is a dict with id, name and path; segmentations also carry
    kind ('segmentation' or 'labelmap') and ref_volume_id (the scan the
    segmentation was drawn on, when the scene records it).
    """
    volumes, segmentations = [], []
    mrml_files = sorted(
        (os.path.join(dp, f) for dp, _, fs in os.walk(scene_dir)
         for f in fs if f.lower().endswith(".mrml")),
        key=lambda p: (p.count(os.sep), p))
    if mrml_files:
        try:
            mrml_dir = os.path.dirname(mrml_files[0])
            root = ET.parse(mrml_files[0]).getroot()
            nodes = {n.get("id"): n for n in root if n.get("id")}

            def stored_file(node):
                for sid in _mrml_refs(node).get("storage", []):
                    fname = nodes[sid].get("fileName") if sid in nodes else None
                    if fname:
                        fname = unquote(fname)
                        path = os.path.normpath(os.path.join(mrml_dir, fname))
                        if os.path.isfile(path):
                            return path
                        found = _find_file(scene_dir, os.path.basename(fname))
                        if found:
                            return found
                return None

            # The Segment Editor also records which scan a segmentation belongs to.
            editor_source = {}
            for n in root:
                if n.tag == "SegmentEditor":
                    r = _mrml_refs(n)
                    src = r.get("sourceVolumeRef") or r.get("masterVolumeRef")
                    for seg_id in r.get("segmentationRef", []):
                        if src:
                            editor_source[seg_id] = src[0]

            for n in root:
                if n.tag not in ("Volume", "LabelMapVolume", "Segmentation"):
                    continue
                path = stored_file(n)
                if not path:
                    continue
                item = {"id": n.get("id"), "name": n.get("name") or "", "path": path}
                if n.tag == "Segmentation":
                    ref = (_mrml_refs(n).get("referenceImageGeometryRef") or [None])[0]
                    item.update(kind="segmentation",
                                ref_volume_id=ref or editor_source.get(n.get("id")))
                    segmentations.append(item)
                elif n.tag == "LabelMapVolume" or n.get("labelMap") == "1":
                    item.update(kind="labelmap", ref_volume_id=None)
                    segmentations.append(item)
                else:
                    volumes.append(item)
            return volumes, segmentations
        except Exception as e:
            warnings.warn(f"  Could not read the MRB scene file ({e}); "
                          "looking at its data files directly.")
            volumes, segmentations = [], []

    # Fallback without a readable scene file: sort the data files by name.
    for dp, _, fs in os.walk(scene_dir):
        for f in sorted(fs):
            path, low = os.path.join(dp, f), f.lower()
            if low.endswith(".seg.nrrd"):
                segmentations.append({"id": None, "name": f[:-9], "path": path,
                                      "kind": "segmentation", "ref_volume_id": None})
            elif low.endswith(IMAGE_FILE_ENDINGS):
                stem = f.split(".")[0]
                if stem.lower().endswith(("-label", "_label")):
                    segmentations.append({"id": None, "name": stem, "path": path,
                                          "kind": "labelmap", "ref_volume_id": None})
                else:
                    volumes.append({"id": None, "name": stem, "path": path})
    return volumes, segmentations

def pick_reference_volume(seg, volumes):
    """Choose the scan a segmentation belongs to. Returns (scan, reason)."""
    by_id = {v["id"]: v for v in volumes if v["id"]}
    if seg.get("ref_volume_id") in by_id:
        return by_id[seg["ref_volume_id"]], "the scan it was drawn on"
    if len(volumes) == 1:
        return volumes[0], "the only scan in the MRB"
    grids = {v["path"]: image_grid(v["path"]) for v in volumes}
    seg_grid = image_grid(seg["path"])
    same_grid = [v for v in volumes if grids[v["path"]][1:] == seg_grid[1:]]
    pool = same_grid or volumes
    best = max(pool, key=lambda v: np.prod(grids[v["path"]][0]))
    if same_grid:
        return best, "largest scan with the segmentation's voxel spacing"
    warnings.warn("  The MRB does not say which scan the segmentation belongs to; "
                  "using the largest scan. Check this mask visually.")
    return best, "largest scan; no better clue found"

def box_fill(mask):
    """Fraction of the segment's bounding box that the segment fills."""
    count = int(np.count_nonzero(mask))
    if count == 0:
        return 0, 0.0
    box = 1
    for axis in range(mask.ndim):
        other = tuple(a for a in range(mask.ndim) if a != axis)
        hits = np.flatnonzero(mask.any(axis=other))
        box *= int(hits[-1] - hits[0] + 1)
    return count, count / box

def read_segments_on_scan(seg, img):
    """Read every segment and place it on the scan's voxel grid.

    Returns (segments, layers): layers are numpy arrays (z, y, x) on the scan
    grid; each segment dict says which layer and label value it uses.
    """
    reader = sitk.ImageFileReader()
    reader.SetFileName(seg["path"])
    seg_img = reader.Execute()
    meta = {k: reader.GetMetaData(k) for k in reader.GetMetaDataKeys()}

    # Overlapping segments are stored in separate layers.
    n_layers = seg_img.GetNumberOfComponentsPerPixel()
    native = ([seg_img] if n_layers == 1 else
              [sitk.VectorIndexSelectionCast(seg_img, i) for i in range(n_layers)])
    # The segmentation may cover a smaller box than the scan; put it on the
    # scan's grid (nearest neighbour keeps the label values unchanged).
    layers = [sitk.GetArrayFromImage(
                  sitk.Resample(layer, img, sitk.Transform(),
                                sitk.sitkNearestNeighbor, 0, layer.GetPixelID()))
              for layer in native]
    native_voxel_ml = np.prod(seg_img.GetSpacing()) / 1000.0
    scan_voxel_ml = np.prod(img.GetSpacing()) / 1000.0

    segments = []
    i = 0
    while f"Segment{i}_ID" in meta or f"Segment{i}_Name" in meta:
        name = meta.get(f"Segment{i}_Name") or meta.get(f"Segment{i}_ID") or f"Segment_{i + 1}"
        segments.append({
            "name": name,
            "layer": int(meta.get(f"Segment{i}_Layer", i if n_layers > 1 else 0)),
            "label": int(meta.get(f"Segment{i}_LabelValue", 1)),
            "auto_named": (meta.get(f"Segment{i}_NameAutoGenerated") == "1"
                           or bool(DEFAULT_SEGMENT_NAME.fullmatch(name.strip()))),
        })
        i += 1

    if not segments:
        # A plain labelmap has no segment names: each non-zero value is one segment.
        arr0 = layers[0]
        values = [int(v) for v in np.unique(arr0[arr0 != 0])]
        base = seg["name"] or os.path.basename(seg["path"])
        for v in values:
            segments.append({"name": base if len(values) == 1 else f"{base} {v}",
                             "layer": 0, "label": v, "auto_named": True})

    for s in segments:
        if s["layer"] >= len(layers):
            s.update(voxels=0, fill=0.0, kept=1.0)
            continue
        s["voxels"], s["fill"] = box_fill(layers[s["layer"]] == s["label"])
        native_count = int(np.count_nonzero(
            sitk.GetArrayViewFromImage(native[s["layer"]]) == s["label"]))
        native_ml = native_count * native_voxel_ml
        s["kept"] = (s["voxels"] * scan_voxel_ml / native_ml) if native_ml else 1.0
    return segments, layers

def choose_mrb_segments(segments):
    """Pick the tumour segment(s). Returns (chosen segments, reason)."""
    for s in segments:
        s["status"] = "empty" if s["voxels"] == 0 else "not used"
    present = [s for s in segments if s["voxels"] > 0]

    chosen = [s for s in present if is_target_roi(s["name"])]
    reason = "segment name matches the tumour name rules"
    if not chosen and not MRB_PICK_UNNAMED_SEGMENT:
        reason = "no segment name matches the tumour name rules"
    elif not chosen:
        unnamed = [s for s in present if s["auto_named"]]
        rest = []
        for s in unnamed:
            if s["fill"] >= MRB_BACKGROUND_FILL:
                s["status"] = "skipped (box-shaped background)"
            else:
                rest.append(s)
        if len(rest) == 1:
            chosen = rest
            reason = ("no name matched; used the only default-named segment "
                      "that is not a box-shaped background")
        elif len(rest) > 1:
            reason = (f"no name matched, and {len(rest)} default-named segments could "
                      "be the tumour, so none was used. Rename the tumour segment in "
                      "Slicer, or add its name to the name rules")
        else:
            reason = "no segment name matches the tumour name rules"
    for s in chosen:
        s["status"] = "TUMOUR (mask label 1)"
    return chosen, reason

# Saving and bookkeeping (shared by DICOM and MRB input)

# Output files are named by patient ID only: <patient>.nrrd and <patient>_mask.nrrd.
counts = {"paired_dicom": 0, "paired_mrb": 0, "unpaired": 0}
paired_patients = set()   # patients that already have a scan-mask pair
reserved_outputs = set()  # files written in this run (never overwritten)
skipped_extra = []        # (patient, file) skipped because the patient already has a pair
no_tumour_found = []      # (file, names found) for the summary at the end

def already_paired(patient_key, source_file):
    """True (and note it) if this patient already has a scan-mask pair."""
    if patient_key in paired_patients:
        print(f"  Patient {patient_key} already has a scan-mask pair; "
              f"skipping {os.path.basename(source_file)}.")
        skipped_extra.append((patient_key, source_file))
        return True
    return False

def save_unpaired(img, patient_key, note):
    out_img = os.path.join(output_root_unpaired, f"{patient_key}.nrrd")
    if os.path.abspath(out_img) in reserved_outputs:
        print(f"  {note}; an image-only file for {patient_key} was already saved in this run, "
              "so this one is not written.")
        return
    try:
        _save_nrrd(img, out_img, compress=True)
        reserved_outputs.add(os.path.abspath(out_img))
        counts["unpaired"] += 1
        print(f"  {note}; saved image separately: {out_img}")
    except Exception as e:
        warnings.warn(f"  Failed to save unpaired image NRRD: {e}")

def save_outputs(img, label_arr, patient_key, roi_names, source):
    """Save scan + mask, or only the scan if the mask is empty.

    Returns True when a scan-mask pair was saved.
    """
    if not np.any(label_arr):
        save_unpaired(img, patient_key, "Empty or unmatched mask")
        return False
    out_img = os.path.join(output_root, f"{patient_key}.nrrd")
    out_mask = os.path.join(output_root, f"{patient_key}_mask.nrrd")
    output_keys = {os.path.abspath(out_img), os.path.abspath(out_mask)}
    if reserved_outputs & output_keys:
        warnings.warn(f"  Duplicate output name for {patient_key}; skipping to prevent overwrite.")
        return False
    try:
        mask_img = numpy_to_sitk_like(img, label_arr)
        mask_img = sitk.Cast(mask_img, sitk.sitkUInt16)
        _save_nrrd(img, out_img, compress=True)
        _save_nrrd(mask_img, out_mask, compress=True)
    except Exception as e:
        warnings.warn(f"  Failed to save NRRD pair: {e}")
        return False
    reserved_outputs.update(output_keys)
    counts["paired_" + source] += 1
    paired_patients.add(patient_key)
    print(f"  Saved pair: {out_img} | {out_mask}")
    print("  ROI labels:", {n: 1 for n in roi_names})
    return True

# One patient, DICOM input

def process_dicom_patient(patient_dir, all_dcms, skip_unpaired_fallback=False):
    patient_key = os.path.basename(os.path.normpath(patient_dir))
    series_map    = index_series_files_by_uid(all_dcms)
    rtstruct_list = list_rtstruct_files(patient_dir)

    print(f"  Series: {len(series_map)} | RTSTRUCTs: {len(rtstruct_list)} | DICOM files: {len(all_dcms)}")

    # Without RTSTRUCT, keep the image outside Notebook 2's paired-input folder.
    if not rtstruct_list:
        if skip_unpaired_fallback:
            print("  No RTSTRUCT; this patient's MRB already gave a scan-mask pair, "
                  "so no extra image-only file is written.")
            return
        warnings.warn(f"No RTSTRUCT found under {patient_dir}; exporting an unpaired image.")
        fb_uid = choose_fallback_uid(series_map)
        if fb_uid:
            try:
                files_sorted = sort_slice_files(series_map[fb_uid])
                img = load_sitk_from_files(files_sorted)
                save_unpaired(img, patient_key, "No RTSTRUCT")
            except Exception as e:
                warnings.warn(f"Failed to export fallback image for {patient_dir}: {e}")
        else:
            warnings.warn(f"No DICOM series found in {patient_dir}.")
        return

    # Process each RTSTRUCT.
    for rs_idx, rtstruct_path in enumerate(rtstruct_list, 1):
        print(f"  RTSTRUCT [{rs_idx}/{len(rtstruct_list)}]: {rtstruct_path}")
        if already_paired(patient_key, rtstruct_path):
            continue

        # Match the RTSTRUCT to its image series.
        ref_uids = get_referenced_series_uids_from_rtstruct(rtstruct_path)
        chosen_uid = choose_uid_with_most_files(ref_uids, series_map) if ref_uids else None
        if chosen_uid is None:
            warnings.warn("  Referenced SeriesInstanceUID not found locally; using the largest series as fallback.")
            chosen_uid = choose_fallback_uid(series_map)
        if chosen_uid is None:
            warnings.warn("  No DICOM series found in patient; skipping this RTSTRUCT.")
            continue

        files = series_map.get(chosen_uid, [])
        if not files:
            warnings.warn(f"  No files for chosen series UID {chosen_uid}; skipping this RTSTRUCT.")
            continue

        files_sorted = sort_slice_files(files)

        # Load the image volume.
        try:
            img = load_sitk_from_files(files_sorted)
        except Exception as e:
            warnings.warn(f"  Failed to load image series (UID {chosen_uid}): {e}")
            continue

        # Isolate the selected series for rt-utils.
        tmp_series_dir = None
        try:
            tmp_series_dir = make_temp_series_dir(files_sorted)
            rtstruct = RTStructBuilder.create_from(
                dicom_series_path=tmp_series_dir,
                rt_struct_path=rtstruct_path
            )
        except Exception as e:
            warnings.warn(f"  RTStructBuilder failed on {rtstruct_path}: {e}")
            if tmp_series_dir and os.path.isdir(tmp_series_dir):
                shutil.rmtree(tmp_series_dir, ignore_errors=True)
            continue

        # Keep matching tumour ROIs and exclude skull ROIs.
        roi_names = rtstruct.get_roi_names() or []

        target_roi_names = [n for n in roi_names if is_target_roi(n)]

        if not target_roi_names:
            print("  No meningioma-like ROI name found (after excluding SKULL). "
                  "Mask will not be written for this RTSTRUCT.")
            no_tumour_found.append((rtstruct_path, list(roi_names)))


        # Build a binary tumour mask in image array order (z, y, x).
        arr_shape = sitk.GetArrayFromImage(img).shape  # (z, y, x)
        label_arr = np.zeros(arr_shape, dtype=np.uint16)

        if target_roi_names:
            for roi_name in target_roi_names:
                try:
                    roi_mask = rtstruct.get_roi_mask_by_name(roi_name)  # often (y,x,z)
                    if roi_mask is None:
                        print(f"    ROI '{roi_name}' returned None; skipping.")
                        continue
                    aligned_mask, how = align_mask_axes(roi_mask, arr_shape)
                    if aligned_mask is None:
                        warnings.warn(
                            f"    ROI '{roi_name}' mask shape {roi_mask.shape} "
                            f"could not be aligned to image shape {arr_shape}; skipping."
                        )
                        continue
                    if how != "identity":
                        print(f"    ROI '{roi_name}': applied {how} to match (z,y,x)")
                    label_arr[aligned_mask > 0] = 1
                    print(f"    Added MENINGIOMA ROI '{roi_name}' (label 1)")
                except Exception as e:
                    warnings.warn(f"    Failed to rasterize ROI '{roi_name}': {e}")
        else:
            print("  MENINGIOMA ROI not found; no mask will be written.")

        save_outputs(img, label_arr, patient_key, target_roi_names, "dicom")

        # Remove the temporary series directory.
        if tmp_series_dir and os.path.isdir(tmp_series_dir):
            shutil.rmtree(tmp_series_dir, ignore_errors=True)

# One MRB file

def process_mrb(mrb_path, patient_key):
    """Convert one MRB file. Returns the number of scan-mask pairs saved."""
    print(f"  MRB: {mrb_path}")
    if already_paired(patient_key, mrb_path):
        return 0
    pairs = 0
    tmp_dir = tempfile.mkdtemp(prefix="mrb_")
    try:
        try:
            with zipfile.ZipFile(mrb_path) as zf:
                zf.extractall(tmp_dir)
        except Exception as e:
            warnings.warn(f"  Could not unpack {mrb_path} (incomplete or not an MRB?): {e}")
            return 0

        volumes, segmentations = read_mrb_scene(tmp_dir)
        print(f"  Scans: {len(volumes)} | Segmentations: {len(segmentations)}")
        if not volumes:
            warnings.warn(f"  No scan found inside {mrb_path}; skipping.")
            return 0

        if not segmentations:
            warnings.warn(f"  No segmentation inside {mrb_path}; exporting an unpaired image.")
            vol = max(volumes, key=lambda v: np.prod(image_grid(v["path"])[0]))
            save_unpaired(sitk.ReadImage(vol["path"]), patient_key, "No segmentation")
            no_tumour_found.append((mrb_path, []))
            return 0

        for s_idx, seg in enumerate(segmentations, 1):
            if already_paired(patient_key, seg["path"]):
                continue
            vol, why = pick_reference_volume(seg, volumes)
            print(f"  Segmentation [{s_idx}/{len(segmentations)}] '{seg['name']}' "
                  f"on scan '{vol['name']}' ({why})")
            try:
                img = sitk.ReadImage(vol["path"])
            except Exception as e:
                warnings.warn(f"  Failed to read scan {vol['path']}: {e}")
                continue
            if img.GetDimension() != 3 or img.GetNumberOfComponentsPerPixel() != 1:
                warnings.warn(f"  Scan '{vol['name']}' is not a 3D greyscale volume; skipping.")
                continue
            try:
                segments, layers = read_segments_on_scan(seg, img)
            except Exception as e:
                warnings.warn(f"  Failed to read segmentation {seg['path']}: {e}")
                save_unpaired(img, patient_key, "Unreadable segmentation")
                continue

            chosen, reason = choose_mrb_segments(segments)
            for s in segments:
                print(f"    '{s['name']}': {s['voxels']:,} voxels, fills {s['fill']:.0%} "
                      f"of its box -> {s['status']}")
                if s["voxels"] and s["kept"] < 0.95:
                    warnings.warn(f"    Only {s['kept']:.0%} of segment '{s['name']}' lies "
                                  "inside the scan; the rest was cut off.")
            print(f"  Decision: {reason}.")
            if not chosen:
                no_tumour_found.append((mrb_path, [s["name"] for s in segments]))

            label_arr = np.zeros(layers[0].shape, dtype=np.uint16)
            for s in chosen:
                label_arr[layers[s["layer"]] == s["label"]] = 1
            if save_outputs(img, label_arr, patient_key,
                            [s["name"] for s in chosen], "mrb"):
                pairs += 1
    except Exception as e:
        warnings.warn(f"  Failed to convert {mrb_path}: {e}")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    return pairs

# Main conversion

# Use every configured root that exists.
configured_dicom_roots = [dicom_root, dicom_root2, dicom_root3]
available_dicom_roots = []
seen_dicom_roots = set()

print("\nChecking configured input folders (DICOM and/or MRB):")
for folder_number, raw_root in enumerate(configured_dicom_roots, 1):
    raw_root = "" if raw_root is None else str(raw_root).strip()
    if not raw_root:
        print(f"  Folder {folder_number}: NOT SET (ignored)")
        continue

    root = os.path.abspath(os.path.expanduser(raw_root))
    if root in seen_dicom_roots:
        print(f"  Folder {folder_number}: DUPLICATE PATH (ignored) -> {root}")
        continue
    seen_dicom_roots.add(root)

    if os.path.isdir(root):
        available_dicom_roots.append(root)
        print(f"  Folder {folder_number}: FOUND -> {root}")
    else:
        print(f"  Folder {folder_number}: NOT FOUND (ignored) -> {root}")

if not available_dicom_roots:
    raise FileNotFoundError(
        "None of the three configured input folders could be found. "
        "Check dicom_root, dicom_root2 and dicom_root3, then run again."
    )

print(f"\nStarting conversion with {len(available_dicom_roots)} available input folder(s).")

# Each patient is a folder, or a .mrb file placed directly in a root.
cases = []  # (patient name, path)
for root in available_dicom_roots:
    n_folders = n_mrb = 0
    for entry in sorted(os.listdir(root)):
        path = os.path.join(root, entry)
        if os.path.isdir(path):
            cases.append((entry, path))
            n_folders += 1
        elif os.path.isfile(path) and entry.lower().endswith(".mrb"):
            cases.append((os.path.splitext(entry)[0], path))
            n_mrb += 1
    print(f"  {root}: {n_folders} patient folder(s), {n_mrb} MRB file(s) directly in the folder")

cases.sort(key=lambda c: c[1])
if not cases:
    raise FileNotFoundError(
        'The available roots contain no patient folders and no .mrb files. '
        'Each root must contain one folder or one .mrb file per patient.'
    )

patient_names = {}
for name, path in cases:
    patient_names.setdefault(name, []).append(path)
duplicate_names = {k: v for k, v in patient_names.items() if len(v) > 1}
if duplicate_names:
    details = '; '.join(f'{k}: {v}' for k, v in sorted(duplicate_names.items()))
    raise ValueError(
        'Duplicate patient names were found across the roots (folder names and '
        '.mrb file names without ".mrb" must be unique). '
        f'Rename or merge them to prevent output collisions: {details}'
    )

print(f"Total patients queued for conversion: {len(cases)}")

for p_idx, (patient_key, path) in enumerate(cases, 1):
    print(f"\n[{p_idx}/{len(cases)}] Patient: {path}")

    if os.path.isfile(path):  # a .mrb file directly in a root
        print("  Format detected: MRB")
        process_mrb(path, patient_key)
        continue

    mrb_files = list_mrb_files(path)
    all_dcms = list_all_dicoms(path)
    found = [label for label, files in (("MRB", mrb_files), ("DICOM", all_dcms)) if files]
    print(f"  Format detected: {' + '.join(found) if found else 'none'}")

    mrb_pairs = sum(process_mrb(m, patient_key) for m in mrb_files)
    if all_dcms:
        process_dicom_patient(path, all_dcms, skip_unpaired_fallback=mrb_pairs > 0)
    elif not mrb_files:
        warnings.warn(f"No DICOM or MRB files found in {path}.")

if skipped_extra:
    print("\nThese patients had more than one outline/segmentation file. Only the first "
          "scan-mask pair was saved; check that it is the one you want:")
    for patient_key, source_file in skipped_extra:
        print(f"  {patient_key}: skipped {source_file}")
if no_tumour_found:
    print("\nNo tumour outline or segment was recognised in these files.")
    print("If one of the names listed is the tumour, add it (in lowercase) to "
          "MENINGIOMA_KEYWORDS or SHORT_EXACT_CODES and run again:")
    for source_file, names in no_tumour_found:
        print(f"  {source_file}: {names if names else 'no outlines/segments'}")
n_paired = counts["paired_dicom"] + counts["paired_mrb"]
print(f"\nDone: {n_paired} paired (DICOM: {counts['paired_dicom']}, MRB: {counts['paired_mrb']}) "
      f"and {counts['unpaired']} image-only output(s).")